# SFT Training Template

This notebook helps you get started with **Supervised Fine-Tuning (SFT)** using Training Hub.
It auto-detects your GPU hardware and calculates safe hyperparameters so you can launch
a training job with minimal configuration.

## What you need
1. A HuggingFace model path (or local model)
2. A training dataset in JSONL chat format
3. One or more GPUs

## What this notebook does
- Detects available GPUs and VRAM
- Calculates memory-safe values for `max_tokens_per_gpu` and `nproc_per_node`
- Provides sensible defaults for learning rate, batch size, and epochs
- Launches training with a single call to `training_hub.sft()`

## Setup

In [ ]:
# Install training-hub with SFT/CUDA support (two-step install required)
# Skip if already installed
# %pip install training-hub
# %pip install training-hub[cuda] --no-build-isolation

## 1. User Inputs

Configure these four values. Everything else is calculated automatically.

In [ ]:
# === REQUIRED: Set these ===
MODEL_PATH = "ibm-granite/granite-3.3-8b-instruct"  # HuggingFace model or local path
DATA_PATH = "./train_data.jsonl"                     # Path to your JSONL training data
CKPT_OUTPUT_DIR = "./sft_checkpoints"                # Where to save checkpoints

# === OPTIONAL: Hardware overrides ===
NUM_GPUS = None       # Auto-detected if None
NUM_NODES = 1         # Number of nodes (default: 1)
VRAM_PER_GPU = None   # Auto-detected if None (in GB, e.g. 80)

## 2. Hardware Detection

In [ ]:
import torch
from transformers import AutoConfig

# Detect GPUs
if NUM_GPUS is None:
    NUM_GPUS = torch.cuda.device_count()
    if NUM_GPUS <= 0:
        raise RuntimeError("No GPUs detected. Set NUM_GPUS manually.")

# Detect VRAM
if VRAM_PER_GPU is None:
    if not torch.cuda.is_available():
        raise RuntimeError("No CUDA device found. Set VRAM_PER_GPU manually (in GB).")
    props = torch.cuda.get_device_properties(0)
    VRAM_PER_GPU = props.total_memory / (1024**3)  # bytes to GB

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A (manual config)"
print(f"GPU: {gpu_name}")
print(f"GPUs: {NUM_GPUS}")
print(f"Nodes: {NUM_NODES}")
print(f"VRAM per GPU: {VRAM_PER_GPU:.1f} GB")
print(f"Total VRAM: {NUM_GPUS * NUM_NODES * VRAM_PER_GPU:.1f} GB")

## 3. Calculate Safe Hyperparameters

SFT trains in FP32 with mixed precision. Memory per GPU is dominated by:
- **Model parameters**: 4 bytes each (FP32)
- **Gradients**: 4 bytes each
- **AdamW optimizer states**: 8 bytes each (2 states × 4 bytes)
- **Activations**: scales with `max_tokens_per_gpu`

Total base memory ≈ `num_params × 16 bytes / num_gpus`. The remaining VRAM
determines how large `max_tokens_per_gpu` can be.

In [ ]:
# Load model config (lightweight — no model download)
config = AutoConfig.from_pretrained(MODEL_PATH)
hidden = config.hidden_size
layers = config.num_hidden_layers
intermediate = getattr(config, "intermediate_size", hidden * 4)
n_heads = config.num_attention_heads
n_kv = getattr(config, "num_key_value_heads", n_heads)
head_dim = hidden // n_heads

attn = hidden * (n_heads + 2 * n_kv) * head_dim + hidden * hidden
mlp = 3 * hidden * intermediate
num_params = config.vocab_size * hidden + layers * (attn + mlp)

bytes_per_param = 16  # param(4) + grad(4) + adamw(8)
base_memory_per_gpu_gb = (num_params * bytes_per_param) / (NUM_GPUS * 1024**3)

# Available VRAM for activations (leave 15% headroom for CUDA overhead)
usable_vram = VRAM_PER_GPU * 0.85
if base_memory_per_gpu_gb > usable_vram:
    print(f"WARNING: Base memory ({base_memory_per_gpu_gb:.1f} GB) exceeds usable VRAM ({usable_vram:.1f} GB).")
    print("Add more GPUs or use a smaller model.")
activation_budget_gb = max(usable_vram - base_memory_per_gpu_gb, 1.0)

# Activation memory scales roughly as: hidden_size * num_layers * max_tokens_per_gpu * 2 bytes
# Solve for max_tokens_per_gpu given our budget
bytes_per_token = hidden * layers * 2  # rough activation cost per token
max_tokens_from_budget = int((activation_budget_gb * 1024**3) / bytes_per_token)

# Clamp to reasonable power-of-2 aligned values
SAFE_MAX_TOKENS_PER_GPU = min(max(256, (max_tokens_from_budget // 256) * 256), 16384)
SAFE_MAX_SEQ_LEN = min(SAFE_MAX_TOKENS_PER_GPU, 4096)  # max_seq_len <= max_tokens_per_gpu

print(f"\nModel: {MODEL_PATH}")
print(f"Estimated parameters: {num_params / 1e9:.2f}B")
print(f"Base memory per GPU: {base_memory_per_gpu_gb:.1f} GB")
print(f"Activation budget per GPU: {activation_budget_gb:.1f} GB")
print(f"\nCalculated safe values:")
print(f"  max_tokens_per_gpu = {SAFE_MAX_TOKENS_PER_GPU}")
print(f"  max_seq_len = {SAFE_MAX_SEQ_LEN}")
print(f"  nproc_per_node = {NUM_GPUS}")

## 4. Training Configuration

These defaults work well for most fine-tuning tasks. Adjust as needed.

In [ ]:
# Training hyperparameters
LEARNING_RATE = 5e-6          # Lower (1e-6) for large datasets, higher (5e-6) for small
EFFECTIVE_BATCH_SIZE = 64     # 32-64 for <1k samples, 128 for 1k-10k, 256+ for >10k
NUM_EPOCHS = 2                # 2-3 typical; more risks overfitting

print("Training config:")
print(f"  learning_rate = {LEARNING_RATE}")
print(f"  effective_batch_size = {EFFECTIVE_BATCH_SIZE}")
print(f"  num_epochs = {NUM_EPOCHS}")
print(f"  max_seq_len = {SAFE_MAX_SEQ_LEN}")
print(f"  max_tokens_per_gpu = {SAFE_MAX_TOKENS_PER_GPU}")
print(f"  nproc_per_node = {NUM_GPUS}")
if NUM_NODES > 1:
    print(f"  nnodes = {NUM_NODES}")

## 5. Launch Training

In [ ]:
from training_hub import sft

result = sft(
    model_path=MODEL_PATH,
    data_path=DATA_PATH,
    ckpt_output_dir=CKPT_OUTPUT_DIR,
    data_output_dir=CKPT_OUTPUT_DIR + "/processed_data",
    # Calculated values
    max_tokens_per_gpu=SAFE_MAX_TOKENS_PER_GPU,
    max_seq_len=SAFE_MAX_SEQ_LEN,
    nproc_per_node=NUM_GPUS,
    nnodes=NUM_NODES if NUM_NODES > 1 else None,
    # Training hyperparameters
    learning_rate=LEARNING_RATE,
    effective_batch_size=EFFECTIVE_BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
)

print("Training complete!")

## 6. Visualize Training Loss

In [ ]:
from training_hub import plot_loss

plot_loss(CKPT_OUTPUT_DIR, ema=True)

## Tips

- **OOM?** Reduce `SAFE_MAX_TOKENS_PER_GPU` or enable Liger kernels by adding `use_liger=True` to the `sft()` call
- **Slow convergence?** Increase `LEARNING_RATE` (up to 1e-5) or `NUM_EPOCHS`
- **Overfitting?** Reduce `NUM_EPOCHS` or increase `EFFECTIVE_BATCH_SIZE`
- **Knowledge data?** Add `unmask=true` to your JSONL samples to unmask user messages for better knowledge ingestion
- **Pretraining on documents?** Add `is_pretraining=True` and `block_size=2048` to the `sft()` call